# CLIP-style training on Colab

Fresh run with config baked into the notebook. Edit `PROJECT_DIR` below to point at where you've uploaded the repo on Drive. Checkpoints + metrics are written back to Drive so they survive runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q --upgrade 'torch>=2.10' 'transformers>=4.40' 'datasets>=2.18' tqdm matplotlib

In [ ]:
import os, sys

# Edit this if your project lives elsewhere on Drive.
PROJECT_DIR = '/content/drive/MyDrive/nlp_vae_memory_module'

assert os.path.isdir(PROJECT_DIR), f'project dir not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
print('cwd:', os.getcwd())
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
from config import Config

cfg = Config()

# --- model ---
cfg.d_model = 512
cfg.n_layers = 8
cfg.n_heads = 8
cfg.ffn_dim = 2048
cfg.dropout = 0.0
cfg.d_sine = 4

# --- signal ---
cfg.n_samples = 2048
cfg.duration = 1.0
cfg.f_min = 1.0
cfg.f_max = 960.0
cfg.f_bias_spread = 3.0
cfg.A_max = 10.0
cfg.freq_sep_min_bins = 4.0
cfg.freq_sep_lambda = 0.05

# --- CLIP training ---
cfg.clip_dataset_name = 'sentence-transformers/all-nli'
cfg.clip_dataset_config = 'pair'
cfg.clip_max_len = 64
cfg.clip_val_frac = 0.05
cfg.clip_batch_size = 256
cfg.clip_grad_accum_steps = 1
cfg.clip_lr = 1e-4
cfg.clip_warmup_steps = 1500
cfg.clip_max_steps = 40000
cfg.clip_logit_scale_init = 2.6593
cfg.clip_logit_scale_max = 4.6052
cfg.clip_log_every = 50
cfg.clip_val_every = 1000
cfg.clip_val_batches = 50
cfg.clip_ckpt_every = 2000

# Persist checkpoints to Drive so they survive disconnects.
cfg.clip_ckpt_dir = os.path.join(PROJECT_DIR, 'all-training', 'clip_colab')

cfg.weight_decay = 0.01
cfg.grad_clip = 1.0

# --- runtime ---
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.num_workers = 2
cfg.seed = 0

for k, v in vars(cfg).items():
    print(f'{k:30s} = {v}')

In [ ]:
from run_utils import make_run_dir, save_config
from train_clip import train

os.makedirs(cfg.clip_ckpt_dir, exist_ok=True)
run_dir = make_run_dir(cfg.clip_ckpt_dir)
save_config(cfg, run_dir)
print('run_dir:', run_dir)

train(cfg, run_dir, resume_ckpt=None)

In [ ]:
from run_utils import plot_metrics
plot_metrics(run_dir)
from IPython.display import Image, display
display(Image(os.path.join(run_dir, 'loss.png')))
display(Image(os.path.join(run_dir, 'acc.png')))